In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
import os
import requests


/tmp/ipykernel_19697/3234916412.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [2]:
# The weather tool is sufficient for the task; no web-search backend is required.
search_tool = DuckDuckGoSearchRun()

In [3]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    api_key = os.getenv("OPENWEATHER_API_KEY")
    if not api_key:
        return "OpenWeather API key is not set."

    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except requests.RequestException as exc:
        return f"Weather service request failed for {city}: {exc}"

    data = response.json()
    temp = data['main']['temp']
    description = data['weather'][0]['description']
    return f"The current temperature in {city} is {temp}°C with {description}."

In [4]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)


In [5]:
from langsmith import Client
from langchain_classic.agents import AgentExecutor, create_react_agent

client = Client()
prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True,
)


agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather],
    prompt=prompt,
)
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather],
    verbose=True,
)


In [6]:
response = agent_executor.invoke({"input": "Find out who is chief minister of uttar pradesh and find out the current temperatue in his home town."})
response["output"]




> Entering new AgentExecutor chain...
To find out who the current Chief Minister of Uttar Pradesh is, I will perform a search. After that, I will find the current temperature in his hometown. 

Action: duckduckgo_search  
Action Input: "current Chief Minister of Uttar Pradesh 2023"  Only three chief ministers completed their official tenure of five years: Mayawati, Akhilesh Yadav, and Yogi Adityanath. Yogi Adityanath of the Bharatiya Janata Party is serving as the current and the longest serving Chief Minister of Uttar Pradesh since 19 March 2017. At the age of 26, he became one of the youngest Indian parliamentarians in 1998 and went on to win the next five consecutive terms from Gorakhpur Lok Sabha constituency. [2][9] In 2017, he moved from national to state politics and was appointed chief minister of Uttar Pradesh. [10] 18 hours ago · Complete, updated list of Uttar Pradesh Chief Ministers from 1950 to 2026, with tenure dates, party, and key facts. 18 hours ago · Uttar Pradesh C

'The current Chief Minister of Uttar Pradesh is Yogi Adityanath, and the current temperature in his hometown of Gorakhpur is 21°C with a clear sky.'